In [9]:
!pip install psycopg2-binary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 15.6 MB/s eta 0:00:0000:01m


In [10]:
# Imports & connections
import pandas as pd
import os
import matplotlib.pyplot as plt
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()
DB_URL = os.getenv('DATABASE_URL')
engine = create_engine(DB_URL)

In [ ]:
# SQL query

query = """
SELECT
    apt_icao,
    apt_name,
    state_name,
    SUM(flt_arr_1) AS total_arrivals,
    SUM(dly_apt_arr_1) AS total_delay_min,
    ROUND(SUM(dly_apt_arr_1) / NULLIF(SUM(flt_arr_1), 0), 2) AS avg_delay_per_arrival,
    ROUND(SUM(flt_arr_1_dly)::NUMERIC / NULLIF(SUM(flt_arr_1), 0) * 100, 2) AS pct_delayed,
    ROUND(SUM(flt_arr_1_dly_15)::NUMERIC / NULLIF(SUM(flt_arr_1), 0) * 100, 2) AS pct_delayed_15
FROM fact_airport_delay
WHERE flt_date < '2026-01-01'
GROUP BY apt_icao, apt_name, state_name
HAVING
    SUM(dly_apt_arr_1) IS NOT NULL AND
    SUM(flt_arr_1) > 50000
ORDER BY avg_delay_per_arrival DESC
LIMIT 20;
"""
df_apt = pd.read_sql(query, engine)